# Lab: test a report listing feature
Run from a fresh kernel. All data is synthetic and local.


In [ ]:
import sqlite3
from dataclasses import dataclass
print('Python environment ready')


## Objectives
Build unit and SQLite integration tests, find an ownership/pagination defect, test failure paths, and critique a false-confidence AI test.


## Predict 1 — before running
The baseline slices rows before filtering by owner. If the first two rows belong to another owner and Ana asks for two rows, will Ana see her later report? Write your prediction and reason.


In [ ]:
rows = [
    {'id': 1, 'owner': 'ben', 'title': 'B1'},
    {'id': 2, 'owner': 'ben', 'title': 'B2'},
    {'id': 3, 'owner': 'ana', 'title': 'A1'},
]


In [ ]:
def baseline_list(rows, owner, limit=2, cursor=0):
    page = rows[cursor:cursor + limit]
    visible = [row for row in page if row['owner'] == owner]
    return {'items': visible, 'next': cursor + limit if len(page) == limit else None}
observed = baseline_list(rows, 'ana')
print(observed)
assert observed['items'] == []


The prediction is confirmed: filtering after slicing hides Ana's report. This is the baseline reproduction. **Pre-edit hypothesis:** filtering before pagination will expose the report while preserving owner isolation.


## Predict 2
A limit of 0 is not a useful page. Should the service return an empty page or reject it? Choose a contract before looking at the implementation.


In [ ]:
def validate_page(limit, cursor):
    if not isinstance(limit, int) or limit < 1 or limit > 3:
        return 'limit must be between 1 and 3'
    if not isinstance(cursor, int) or cursor < 0:
        return 'cursor must be non-negative'
    return None
assert validate_page(0, 0) is not None
print('Boundary is rejected before querying')


## Predict 3
If a SQLite connection is closed before a read, should a report endpoint claim success? Predict the safe outcome: a visible dependency error, not an empty successful list.


In [ ]:
def read_sql(conn, owner):
    try:
        return conn.execute('SELECT id, owner, title FROM reports WHERE owner = ? ORDER BY id', (owner,)).fetchall()
    except sqlite3.Error as exc:
        return {'error': type(exc).__name__}
closed = sqlite3.connect(':memory:')
closed.close()
failure = read_sql(closed, 'ana')
print(failure)
assert failure['error'] == 'ProgrammingError'


## Arrange–act–assert unit test
The TODO is executable starter code. Try to write the assertion that proves filtering happens before pagination; then compare with the reference solution below.


In [ ]:
def list_reports(rows, owner, limit=2, cursor=0):
    error = validate_page(limit, cursor)
    if error:
        return {'status': 400, 'error': error}
    visible = [r for r in rows if r['owner'] == owner]
    page = visible[cursor:cursor + limit]
    return {'status': 200, 'items': page, 'next': cursor + limit if len(page) == limit else None}
# Guided TODO: replace this with your own contract assertion.
todo_result = list_reports(rows, 'ana', limit=2)
assert todo_result['status'] == 200  # executable starter check


In [ ]:
# Reference solution: positive, negative, and boundary assertions.
result = list_reports(rows, 'ana', limit=2)
assert result['items'] == [{'id': 3, 'owner': 'ana', 'title': 'A1'}]
assert list_reports(rows, 'ben', limit=2)['items'][0]['owner'] == 'ben'
assert list_reports(rows, 'ana', limit=0)['status'] == 400
assert list_reports(rows, 'ana', limit=4)['status'] == 400
assert list_reports(rows, 'ana', cursor=-1)['status'] == 400
print('Unit contract checks passed')


## Realistic SQLite integration boundary
A fake list cannot catch SQL mistakes. This fixture is fresh for the test and disappears with the connection.


In [ ]:
def make_db():
    conn = sqlite3.connect(':memory:')
    conn.execute('CREATE TABLE reports (id INTEGER PRIMARY KEY, owner TEXT, title TEXT)')
    conn.executemany('INSERT INTO reports VALUES (?, ?, ?)', [(1,'ana','A1'), (2,'ben','B1')])
    conn.commit()
    return conn
db = make_db()
assert read_sql(db, 'ana') == [(1, 'ana', 'A1')]
assert read_sql(db, 'ben') == [(2, 'ben', 'B1')]
db.close()
print('SQLite integration boundary passed and was isolated')


## Intentionally weak AI-style test for critique
This test has false confidence: it checks only a status-like value and never proves ownership or data. Name two missing assertions before reading the explanation.


In [ ]:
def ai_style_test():
    output = list_reports(rows, 'ana')
    assert output['status'] == 200
ai_style_test()
print('The weak test is green, but it would pass if the wrong reports were returned.')


A meaningful replacement checks the owner, fields, ordering, pagination boundary, and failure behavior. This is why a green test count is not enough.


## Independent challenge — attempt before checking
Add a table-driven set of cases for an empty owner, a second page, and a malformed cursor. Write your expected status/items in notes first, then compare with the executable checks below.


In [ ]:
assert list_reports(rows, 'zoe')['items'] == []
many = [{'id': i, 'owner': 'ana', 'title': str(i)} for i in range(4)]
assert list_reports(many, 'ana', 2, 2)['items'][0]['id'] == 2
assert list_reports(rows, 'ana', 2, '0')['status'] == 400
print('Independent challenge passed')


## Exit questions and answers
1. Which test would fail if the owner filter were removed? The unit and SQLite ownership assertions.
2. Why keep SQLite instead of mocking it? To exercise SQL/schema/wiring.
3. What remains unproved? Production DB configuration, load, network, and every input combination.
4. Evidence handoff: save baseline empty-page output, pre-edit hypothesis, passing positive/negative/failure assertions, fixture isolation note, and your AI diff critique.
